# Generation of full MNIST dataset using FRQI encoding

Output is `/mnt/data02/rpotempa/datasets/frqi_mnist_16x16_full.zip`.

## Imports

In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchinfo
from torch.utils import data as torch_data
from torchmetrics import Accuracy

import lightning as L
from lightning.pytorch.loggers import CSVLogger

from qiskit.circuit.library import real_amplitudes

import geqie_qml
from geqie_qml import UnitaryInputLayer, SamplerAnsatzLayer

print("geqie_qml:", geqie_qml.__file__)

geqie_qml: /home/ODM/rpotempa/geqie/geqie-qml/src/geqie_qml/__init__.py


## Configuration

In [2]:
# --- data ---
IMAGE_SIZE = 16               # MNIST is resized to IMAGE_SIZE x IMAGE_SIZE before FRQI encoding
ENCODING = "frqi"
DATASET_SAMPLE_SIZE = 60000      # training images to encode
TEST_SAMPLE_SIZE = 10000
PRECOMPUTED_ZIP = Path("/mnt/data02/rpotempa/datasets/") / f"frqi_mnist_16x16_full.zip"

# --- model ---
N_CLASSES = 10

torch.manual_seed(42)
np.random.seed(42)

## Precompute GEQIE encodings

Encodes a small MNIST subset with FRQI and packages the matrices into a single zip archive
(`train/` and `test/` folders), the layout expected by `load_precomputed_zip_matrices`.
Re-runs are skipped once the archive exists.

In [ ]:
def resize_dataset(data: torch.Tensor, size: int) -> np.ndarray:
    resized = torch.nn.functional.interpolate(
        data.unsqueeze(1).float(),
        size=(size, size),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1)
    return resized.numpy().astype(np.uint8)


def build_precomputed_zip(n_workers) -> None:
    if PRECOMPUTED_ZIP.exists():
        print(f"Reusing existing archive: {PRECOMPUTED_ZIP}")
        return

    mnist_train = torchvision.datasets.MNIST(root="./.data", train=True, download=True)
    mnist_test = torchvision.datasets.MNIST(root="./.data", train=False, download=True)

    train_data = resize_dataset(mnist_train.data[:DATASET_SAMPLE_SIZE], IMAGE_SIZE)
    train_labels = mnist_train.targets[:DATASET_SAMPLE_SIZE]
    test_data = resize_dataset(mnist_test.data[:TEST_SAMPLE_SIZE], IMAGE_SIZE)
    test_labels = mnist_test.targets[:TEST_SAMPLE_SIZE]

    # compute_and_save_circuits skips images whose .npz already exists, so
    # re-running this cell after a dropped connection resumes instead of
    # recomputing everything.
    stage_dir = PRECOMPUTED_ZIP.parent / PRECOMPUTED_ZIP.stem
    geqie_qml.compute_and_save_circuits(
        data=train_data,
        labels=train_labels,
        save_dir=str(stage_dir / "train"),
        geqie_encoding=ENCODING,
        encoding_params={},
        number_of_workers=n_workers,
    )
    geqie_qml.compute_and_save_circuits(
        data=test_data,
        labels=test_labels,
        save_dir=str(stage_dir / "test"),
        geqie_encoding=ENCODING,
        encoding_params={},
        number_of_workers=n_workers,
    )

    # Write to a temp path and rename atomically, so a crash mid-zip never
    # leaves a corrupt archive that a later run would mistake for complete.
    tmp_zip = PRECOMPUTED_ZIP.with_suffix(f".tmp-{os.getpid()}.zip")
    try:
        with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for matrix_file in sorted(stage_dir.glob("*/*.npz")):
                if ".tmp-" in matrix_file.name:
                    continue
                archive.write(matrix_file, arcname=matrix_file.relative_to(stage_dir).as_posix())
        os.replace(tmp_zip, PRECOMPUTED_ZIP)
    finally:
        tmp_zip.unlink(missing_ok=True)
    print(f"Wrote archive: {PRECOMPUTED_ZIP}")


build_precomputed_zip(n_workers=None)


Precomputing 60000 images with encoding 'frqi' using 63 worker(s).


Processing images:   5%|██████▌                                                                                                                                    | 2832/60000 [2:03:27<31:38:48,  1.99s/image]

In [ ]:
train_ds, test_ds = geqie_qml.load_precomputed_zip_matrices(str(PRECOMPUTED_ZIP))

# Infer the qubit count from an actual encoded matrix (2**n x 2**n).
sample_matrix, _ = train_ds[0]
N_QUBITS = int(round(np.log2(sample_matrix.shape[-1])))
print(f"Encoded qubits: {N_QUBITS}")
print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")

Encoded qubits: 9
Train: 737, Val: 184, Test: 288
